# LC 134 — Gas Station
**Day-49 | Greedy Review | Difficulty: Medium**

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px">
<strong>Core Insight:</strong> If total gas >= total cost a
solution always exists. The valid start is always the station
right after the last place your running tank went negative.
</div>

## Official Problem Statement

There are `n` gas stations along a circular route. You are given
two integer arrays `gas` and `cost` where `gas[i]` is the amount
of gas at station `i` and `cost[i]` is the cost to travel from
station `i` to the next station `(i+1) % n`.

Return the starting station's index if you can travel around the
circuit once in the clockwise direction, otherwise return `-1`.
If a solution exists, it is **guaranteed to be unique**.

**Constraints:**
- `1 <= n <= 10^5`
- `0 <= gas[i], cost[i] <= 10^4`

## What This Is Actually Asking

You have a circular track of fuel stations. At each stop you
collect gas but spend some to reach the next stop. You need to
find a starting point where you never run out of gas mid-journey.
The key is that if the total supply is less than total demand,
no start works. Otherwise, exactly one starting point will work,
and greedy logic pinpoints it in a single pass.

## Walk Through an Example by Hand

```
gas  = [1, 2, 3, 4, 5]
cost = [3, 4, 5, 1, 2]
net  = [-2,-2,-2, 3, 3]   (gas[i] - cost[i])

sum(gas)=15 >= sum(cost)=15 => solution exists

tank=0, start=0
i=0: tank += -2 = -2. tank<0 => start=1, tank=0
i=1: tank += -2 = -2. tank<0 => start=2, tank=0
i=2: tank += -2 = -2. tank<0 => start=3, tank=0
i=3: tank += 3  =  3. ok.
i=4: tank += 3  =  6. ok.
Return start = 3

Verify from 3: 4-1=3, 3+5-2=6, 6+1-3=4, 4+2-4=2, 2+3-5=0 ok!
```

## The Picture

```
Circular track (5 stations):

      [0]--(-2)-->[1]--(-2)-->[2]--(-2)-->[3]
       ^                                   |
       |                                (+3)|
      [4]<--(+3)--[4]                       v

net = [-2, -2, -2, +3, +3]

Running tank if we start at 0:
  i=0:  tank= -2  NEGATIVE -> reset start=1
  i=1:  tank= -2  NEGATIVE -> reset start=2
  i=2:  tank= -2  NEGATIVE -> reset start=3
  i=3:  tank= +3  ok
  i=4:  tank= +6  ok
  => start = 3

Intuition: every time tank dips below 0, stations 0..i
CANNOT be the start. The fresh start must be i+1.
```

## When To Use This Pattern

- When you have a **circular traversal** with local gains/losses,
  think greedy start-reset.
- When a problem guarantees **at most one valid answer** exists,
  a greedy single pass can find it without trying all starts.
- When a running sum goes negative, that means everything up to
  the current index is disqualified — reset and move on.
- When global feasibility (sum check) separates the impossible
  case cleanly, do it first before any detailed scan.
- When indices wrap around (mod n), check global conditions
  first, then treat it like a linear scan.

## The Approach

First, if `sum(gas) < sum(cost)`, return -1 immediately — it's
impossible. Otherwise, scan once: maintain a running `tank` and
a candidate `start`. Whenever `tank` drops below zero, all
stations up through the current one are disqualified, so reset
`start = i + 1` and `tank = 0`. After the full pass, `start`
holds the unique valid answer.

In [ ]:
from typing import List

In [ ]:
def test_harness(func):
    cases = [
        # (gas, cost, expected)
        ([1,2,3,4,5], [3,4,5,1,2],  3),   # standard case
        ([2,3,4],     [3,4,3],      -1),  # impossible
        ([5],         [4],           0),  # single station
        ([1,2],       [2,1],         1),  # 2-station ok
        ([1,2],       [2,2],        -1),  # 2-station fail
        ([3,1,1],     [1,2,2],       0),  # start at 0
        ([4,5,2,6,5,3],[3,2,7,3,2,9],-1), # tight fail
    ]
    passed = 0
    for gas, cost, expected in cases:
        result = func(gas, cost)
        status = "PASSED" if result == expected else "FAILED"
        if status == "PASSED":
            passed += 1
        print(f"{status} | gas={gas} cost={cost} "
              f"| expected={expected} | got={result}")
    print(f"\n{passed}/{len(cases)} tests passed.")

test_harness(lambda g, c: None)

In [ ]:
def can_complete_circuit(
    gas: List[int], cost: List[int]
) -> int:
    """
    Find the starting gas station index to complete
    the circular route, or return -1 if impossible.

    Strategy:
        1. If sum(gas) < sum(cost): return -1.
        2. Single pass: track running tank and start.
        3. If tank < 0: reset start=i+1, tank=0.
        4. Return start.

    Args:
        gas:  gas available at each station.
        cost: gas cost to reach next station.

    Returns:
        Starting station index, or -1.
    """
    # TODO: implement solution
    pass

    # Debug hints (remove before final submission):
    # print(f"sum(gas)={sum(gas)}, sum(cost)={sum(cost)}")
    # print(f"net = {[g-c for g,c in zip(gas,cost)]}")
    # print(f"i={i}, tank={tank}, start={start}")
    # print(f"tank<0 -> reset start={i+1}")
    # print(f"Final start={start}")

In [ ]:
# Uncomment and run when solution is ready
# test_harness(can_complete_circuit)

## Complexity

| Approach | Time | Space | Notes |
|---|---|---|---|
| Brute Force | O(n^2) | O(1) | Try every start |
| **Greedy (optimal)** | **O(n)** | **O(1)** | Single pass |

The sum check costs O(n) but is absorbed into the overall O(n).
No extra data structures needed — pure greedy is unbeatable here.

## Real World Connection

At Citi, this maps directly to intraday liquidity management:
each hour is a station where cash inflows (gas) must cover
outflows (cost). The algorithm finds the earliest time window
where you start cash-positive and stay that way all day. In AWS
data pipelines, it models whether a batch job can run from a
given checkpoint without running out of memory/quota mid-run.
For DEs, it's useful when scheduling circular dependency chains
where each job produces and consumes tokens in a ring.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra